# Load Packages

In [ ]:
!pip install torch
!pip install torch-geometric
!pip install pysmiles
!pip install rdkit
# !pip install openai
!pip install openai
!pip install tiktoken
!pip install openai==0.28

import numpy as np
import random
import matplotlib.pyplot as plt
from pysmiles import read_smiles
import pandas as pd
import logging
from tqdm import tqdm
import torch
from torch.nn import Sequential as Seq, Linear, ReLU, CrossEntropyLoss
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, GCNConv
from torch_geometric.utils import remove_self_loops, add_self_loops, degree
from torch_geometric.data import Data
from google.colab import drive
drive.mount('/content/gdrive')
logging.getLogger('pysmiles').setLevel(logging.CRITICAL)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# Load Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import time

# Load the train_test dataset
df = pd.read_csv('gdrive/My Drive/Base_GNN_Solubility/Exp-DB.csv')
df = df.dropna(subset=['logS'])
df.reset_index(drop=True, inplace=True)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Group by unique pairs of SMILES and SMILES_Solvent
group_counts = df.groupby(["solute", "solvent"]).size()
# group_counts = df.groupby(["SMILES"]).size()

# Get unique combinations of SMILES and SMILES_Solvent
unique_combinations = group_counts.index

# Split unique combinations into 60% train, 20% validation, 20% test
train_combinations, temp_combinations = train_test_split(
    unique_combinations, test_size=0.2, random_state=42
)
val_combinations, test_combinations = train_test_split(
    temp_combinations, test_size=0.5, random_state=42
)

# Filter the DataFrame to create train, validation, and test sets
train = df[df.set_index(["solute", "solvent"]).index.isin(train_combinations)]
val = df[df.set_index(["solute", "solvent"]).index.isin(val_combinations)]
test = df[df.set_index(["solute", "solvent"]).index.isin(test_combinations)]

# Debugging: Print the sizes of each split
print("Training Data Size:", train.shape[0])
print("Validation Data Size:", val.shape[0])
print("Test Data Size:", test.shape[0])

Training Data Size: 9887
Validation Data Size: 1236
Test Data Size: 1236


# Featurization

In [ ]:
import os.path as osp
import random
import pandas as pd
import torch
from rdkit import Chem
from tqdm import tqdm
from torch_geometric.data import Data, DataLoader

class GenFeatures:
    def __init__(self):
        self.symbols = [
            'B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'As', 'Se', 'Br',
            'Te', 'I', 'At', 'other'
        ]
        self.hybridizations = [
            Chem.rdchem.HybridizationType.S,
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2,
            Chem.rdchem.HybridizationType.UNSPECIFIED,
            'other',
        ]
        self.stereos = [
            Chem.rdchem.BondStereo.STEREONONE,
            Chem.rdchem.BondStereo.STEREOANY,
            Chem.rdchem.BondStereo.STEREOZ,
            Chem.rdchem.BondStereo.STEREOE,
        ]

    def __call__(self, smiles, i, y):
        # Generate features for the molecule
        data = Data()
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)

        xs = []
        for atom in mol.GetAtoms():
            symbol = [0.] * len(self.symbols)
            atom_symbol = atom.GetSymbol()
            if atom_symbol in self.symbols:
                symbol[self.symbols.index(atom_symbol)] = 1.
            else:
                symbol[-1] = 1.  # Mark as 'other'

            degree = [0.] * 6
            atom_degree = atom.GetDegree()
            if atom_degree < len(degree):
                degree[atom_degree] = 1.

            formal_charge = atom.GetFormalCharge()
            radical_electrons = atom.GetNumRadicalElectrons()
            hybridization = [0.] * len(self.hybridizations)
            try:
                hybridization[self.hybridizations.index(atom.GetHybridization())] = 1.
            except ValueError:
                hybridization[-1] = 1.  # Default to 'other'

            aromaticity = 1. if atom.GetIsAromatic() else 0.
            hydrogens = [0.] * 5
            total_hydrogens = atom.GetTotalNumHs()
            if total_hydrogens < len(hydrogens):
                hydrogens[total_hydrogens] = 1.
            else:
                hydrogens[-1] = 1.

            chirality = 1. if atom.HasProp('_ChiralityPossible') else 0.
            chirality_type = [0.] * 2
            if atom.HasProp('_CIPCode'):
                cip_code = atom.GetProp('_CIPCode')
                if cip_code in ['R', 'S']:
                    chirality_type[['R', 'S'].index(cip_code)] = 1.

            x = torch.tensor(symbol + degree + [formal_charge] +
                             [radical_electrons] + hybridization +
                             [aromaticity] + hydrogens + [chirality] +
                             chirality_type)
            xs.append(x)

        data.x = torch.stack(xs, dim=0)
        data.y = torch.tensor(y, dtype=torch.float)

        edge_indices = []
        edge_attrs = []
        for bond in mol.GetBonds():
            edge_indices += [[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
            edge_indices += [[bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()]]

            bond_type = bond.GetBondType()
            single = 1. if bond_type == Chem.rdchem.BondType.SINGLE else 0.
            double = 1. if bond_type == Chem.rdchem.BondType.DOUBLE else 0.
            triple = 1. if bond_type == Chem.rdchem.BondType.TRIPLE else 0.
            aromatic = 1. if bond_type == Chem.rdchem.BondType.AROMATIC else 0.
            conjugation = 1. if bond.GetIsConjugated() else 0.
            ring = 1. if bond.IsInRing() else 0.
            stereo = [0.] * 4
            stereo[self.stereos.index(bond.GetStereo())] = 1.

            edge_attr = torch.tensor(
                [single, double, triple, aromatic, conjugation, ring] + stereo)

            edge_attrs += [edge_attr, edge_attr]

        if len(edge_attrs) == 0:
            data.edge_index = torch.zeros((2, 0), dtype=torch.long)
            data.edge_attr = torch.zeros((0, 10), dtype=torch.float)
        else:
            data.edge_index = torch.tensor(edge_indices).t().contiguous()
            data.edge_attr = torch.stack(edge_attrs, dim=0)

        graph_level_features = pd.to_numeric(df.iloc[i, -1], errors='coerce')
        graph_level_features = torch.tensor(graph_level_features, dtype=torch.float)


        data.graph_level_features = graph_level_features
        # data.x = torch.cat([data.x, graph_level_features.repeat(data.x.size(0), 1)], dim=-1)


        return data


class Gen_Solvent_Features:
    def __init__(self):
        self.symbols = [
            'B', 'C', 'N', 'O', 'F', 'Si', 'P', 'S', 'Cl', 'As', 'Se', 'Br',
            'Te', 'I', 'At', 'other'
        ]
        self.hybridizations = [
            Chem.rdchem.HybridizationType.S,
            Chem.rdchem.HybridizationType.SP,
            Chem.rdchem.HybridizationType.SP2,
            Chem.rdchem.HybridizationType.SP3,
            Chem.rdchem.HybridizationType.SP3D,
            Chem.rdchem.HybridizationType.SP3D2,
            Chem.rdchem.HybridizationType.UNSPECIFIED,
            'other',
        ]
        self.stereos = [
            Chem.rdchem.BondStereo.STEREONONE,
            Chem.rdchem.BondStereo.STEREOANY,
            Chem.rdchem.BondStereo.STEREOZ,
            Chem.rdchem.BondStereo.STEREOE,
        ]

    def __call__(self, smiles, i, y):
        # Generate features for the molecule
        data = Data()
        mol = Chem.MolFromSmiles(smiles)
        mol = Chem.AddHs(mol)

        xs = []
        for atom in mol.GetAtoms():
            symbol = [0.] * len(self.symbols)
            atom_symbol = atom.GetSymbol()
            if atom_symbol in self.symbols:
                symbol[self.symbols.index(atom_symbol)] = 1.
            else:
                symbol[-1] = 1.  # Mark as 'other'

            degree = [0.] * 6
            atom_degree = atom.GetDegree()
            if atom_degree < len(degree):
                degree[atom_degree] = 1.

            formal_charge = atom.GetFormalCharge()
            radical_electrons = atom.GetNumRadicalElectrons()
            hybridization = [0.] * len(self.hybridizations)
            try:
                hybridization[self.hybridizations.index(atom.GetHybridization())] = 1.
            except ValueError:
                hybridization[-1] = 1.  # Default to 'other'

            aromaticity = 1. if atom.GetIsAromatic() else 0.
            hydrogens = [0.] * 5
            total_hydrogens = atom.GetTotalNumHs()
            if total_hydrogens < len(hydrogens):
                hydrogens[total_hydrogens] = 1.
            else:
                hydrogens[-1] = 1.

            chirality = 1. if atom.HasProp('_ChiralityPossible') else 0.
            chirality_type = [0.] * 2
            if atom.HasProp('_CIPCode'):
                cip_code = atom.GetProp('_CIPCode')
                if cip_code in ['R', 'S']:
                    chirality_type[['R', 'S'].index(cip_code)] = 1.

            x = torch.tensor(symbol + degree + [formal_charge] +
                             [radical_electrons] + hybridization +
                             [aromaticity] + hydrogens + [chirality] +
                             chirality_type)
            xs.append(x)

        data.x = torch.stack(xs, dim=0)
        data.y = torch.tensor(y, dtype=torch.float)

        edge_indices = []
        edge_attrs = []
        for bond in mol.GetBonds():
            edge_indices += [[bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()]]
            edge_indices += [[bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()]]

            bond_type = bond.GetBondType()
            single = 1. if bond_type == Chem.rdchem.BondType.SINGLE else 0.
            double = 1. if bond_type == Chem.rdchem.BondType.DOUBLE else 0.
            triple = 1. if bond_type == Chem.rdchem.BondType.TRIPLE else 0.
            aromatic = 1. if bond_type == Chem.rdchem.BondType.AROMATIC else 0.
            conjugation = 1. if bond.GetIsConjugated() else 0.
            ring = 1. if bond.IsInRing() else 0.
            stereo = [0.] * 4
            stereo[self.stereos.index(bond.GetStereo())] = 1.

            edge_attr = torch.tensor(
                [single, double, triple, aromatic, conjugation, ring] + stereo)

            edge_attrs += [edge_attr, edge_attr]

        if len(edge_attrs) == 0:
            data.edge_index = torch.zeros((2, 0), dtype=torch.long)
            data.edge_attr = torch.zeros((0, 10), dtype=torch.float)
        else:
            data.edge_index = torch.tensor(edge_indices).t().contiguous()
            data.edge_attr = torch.stack(edge_attrs, dim=0)

        graph_level_features = pd.to_numeric(df.iloc[i, -1:], errors='coerce')
        graph_level_features = torch.tensor(graph_level_features, dtype=torch.float)

        data.graph_level_features = graph_level_features
        # data.x = torch.cat([data.x, graph_level_features.repeat(data.x.size(0), 1)], dim=-1)


        return data

'''Train'''
# Assuming df contains columns 'SMILES' for solutes and 'SMILES_Solvent' for solvents
X_smiles = list(train['solute'])
X_smiles_solvent = list(train['solvent'])
Y_solute = list(train['logS'])  # Your target for solutes
Y_solvent = list(train['logS'])   # Your target for solvents

pre_transform_solute = GenFeatures()
pre_transform_solvent = Gen_Solvent_Features()

data_solute = []
data_solvent = []

# Process solutes
for i, smiles in tqdm(enumerate(X_smiles), total=len(X_smiles)):
    try:
        node_edge_featurization = pre_transform_solute(smiles, i, Y_solute[i])
        data_solute.append(node_edge_featurization)
    except Exception as e:
        print(f"Error processing solute {smiles}: {e}")

# Process solvents
for i, smiles in tqdm(enumerate(X_smiles_solvent), total=len(X_smiles_solvent)):
    try:
        node_edge_featurization = pre_transform_solvent(smiles, i, Y_solvent[i])
        data_solvent.append(node_edge_featurization)
    except Exception as e:
        print(f"Error processing solvent {smiles}: {e}")

# Shuffle and split the data
train_data = list(zip(data_solute, data_solvent))


'''Validation'''
# Assuming df contains columns 'SMILES' for solutes and 'SMILES_Solvent' for solvents
X_smiles = list(val['solute'])
X_smiles_solvent = list(val['solvent'])
Y_solute = list(val['logS'])  # Your target for solutes
Y_solvent = list(val['logS'])   # Your target for solvents

pre_transform_solute = GenFeatures()
pre_transform_solvent = Gen_Solvent_Features()

data_solute = []
data_solvent = []

# Process solutes
for i, smiles in tqdm(enumerate(X_smiles), total=len(X_smiles)):
    try:
        node_edge_featurization = pre_transform_solute(smiles, i, Y_solute[i])
        data_solute.append(node_edge_featurization)
    except Exception as e:
        print(f"Error processing solute {smiles}: {e}")

# Process solvents
for i, smiles in tqdm(enumerate(X_smiles_solvent), total=len(X_smiles_solvent)):
    try:
        node_edge_featurization = pre_transform_solvent(smiles, i, Y_solvent[i])
        data_solvent.append(node_edge_featurization)
    except Exception as e:
        print(f"Error processing solvent {smiles}: {e}")

# Shuffle and split the data
val_data = list(zip(data_solute, data_solvent))




'''Test'''
# Assuming df contains columns 'SMILES' for solutes and 'SMILES_Solvent' for solvents
X_smiles = list(test['solute'])
X_smiles_solvent = list(test['solvent'])
Y_solute = list(test['logS'])  # Your target for solutes
Y_solvent = list(test['logS'])   # Your target for solvents

pre_transform_solute = GenFeatures()
pre_transform_solvent = Gen_Solvent_Features()

data_solute = []
data_solvent = []

# Process solutes
for i, smiles in tqdm(enumerate(X_smiles), total=len(X_smiles)):
    try:
        node_edge_featurization = pre_transform_solute(smiles, i, Y_solute[i])
        data_solute.append(node_edge_featurization)
    except Exception as e:
        print(f"Error processing solute {smiles}: {e}")

# Process solvents
for i, smiles in tqdm(enumerate(X_smiles_solvent), total=len(X_smiles_solvent)):
    try:
        node_edge_featurization = pre_transform_solvent(smiles, i, Y_solvent[i])
        data_solvent.append(node_edge_featurization)
    except Exception as e:
        print(f"Error processing solvent {smiles}: {e}")

# Shuffle and split the data
test_data = list(zip(data_solute, data_solvent))

 20%|██        | 2009/9887 [00:13<00:15, 519.90it/s]

Error processing solute CC1=CC=C[NH++]([O-])[CH-]1: Python argument types in
    rdkit.Chem.rdmolops.AddHs(NoneType)
did not match C++ signature:
    AddHs(RDKit::ROMol mol, bool explicitOnly=False, bool addCoords=False, boost::python::api::object onlyOnAtoms=None, bool addResidueInfo=False)
    AddHs(RDKit::ROMol mol, RDKit::MolOps::AddHsParameters params, boost::python::api::object onlyOnAtoms=None)


 77%|███████▋  | 7620/9887 [00:24<00:03, 733.09it/s]

Error processing solute OC(=O)C1=C[NH++]([O-])[CH-]C=C1: Python argument types in
    rdkit.Chem.rdmolops.AddHs(NoneType)
did not match C++ signature:
    AddHs(RDKit::ROMol mol, bool explicitOnly=False, bool addCoords=False, boost::python::api::object onlyOnAtoms=None, bool addResidueInfo=False)
    AddHs(RDKit::ROMol mol, RDKit::MolOps::AddHsParameters params, boost::python::api::object onlyOnAtoms=None)


  0%|          | 0/9887 [00:00<?, ?it/s]/tmp/ipython-input-4274666318.py:227: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  graph_level_features = torch.tensor(graph_level_features, dtype=torch.float)
100%|██████████| 1236/1236 [00:01<00:00, 1168.51it/s]


In [ ]:
from typing import Optional

import torch
import torch.nn.functional as F
from torch import Tensor
from torch.nn import LSTMCell, GRUCell, Linear, Parameter

from torch_geometric.nn import GATConv, GATv2Conv, GCNConv, ResGatedGraphConv, MessagePassing, global_add_pool
from torch_geometric.nn.inits import glorot, zeros
from torch_geometric.typing import Adj, OptTensor
from torch_geometric.utils import softmax
from sklearn.metrics import r2_score



def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
    # Ensure reproducibility in certain GPU operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)




class CrossModalAttention(torch.nn.Module):
    def __init__(self, text_dim, graph_dim, hidden_dim):
        super(CrossModalAttention, self).__init__()
        self.text_linear = torch.nn.Linear(text_dim, hidden_dim)
        self.graph_linear = torch.nn.Linear(graph_dim, hidden_dim)

    def forward(self, text_embeddings, graph_embeddings):
        # Check and reshape text_embeddings to (1, 1, text_dim)
        if text_embeddings.dim() == 2 and text_embeddings.size(0) == 1:  # (1, text_dim)
            text_embeddings = text_embeddings.unsqueeze(1)  # Shape: (1, 1, text_dim)

        text_transformed = self.text_linear(text_embeddings)  # Shape: (1, 1, hidden_dim)

        # Transform graph embeddings
        if graph_embeddings.dim() == 2 and graph_embeddings.size(0) == 1:  # (1, graph_dim)
            graph_embeddings = graph_embeddings.unsqueeze(1)  # Shape: (1, 1, graph_dim)

        # Transform graph embeddings
        graph_transformed = self.graph_linear(graph_embeddings)  # Shape: (1, 1, hidden_dim)

        # We do not need to unsqueeze graph_transformed here.
        # It should be reshaped to (1, hidden_dim) for proper computation
        graph_transformed = graph_transformed.squeeze(1)  # Shape: (1, hidden_dim)

        # Compute attention scores
        attention_scores = torch.bmm(text_transformed, graph_transformed.unsqueeze(2))  # Shape: (1, 1, 1)
        attention_weights = F.softmax(attention_scores, dim=1)  # Normalize over seq_len

        # Apply attention to text embeddings
        context_vector = torch.bmm(attention_weights.transpose(1, 2), text_transformed)  # Shape: (1, 1, hidden_dim)
        context_vector = context_vector.squeeze(1)  # Shape: (1, hidden_dim)

        # Combine context vector with graph embeddings
        combined = torch.cat((context_vector, graph_transformed), dim=1)  # Shape: (1, hidden_dim * 2)



class ProjectionMLP(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ProjectionMLP, self).__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(input_dim, output_dim),
            # torch.nn.ReLU()
            # , torch.nn.Linear(output_dim, output_dim)
        )

    def forward(self, x):
        return self.mlp(x)

class GATEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        edge_dim: int,
        dropout: float = 0.0,
    ):
        super().__init__(aggr='add', node_dim=0)

        self.dropout = dropout

        self.att_l = Parameter(torch.empty(1, out_channels))
        self.att_r = Parameter(torch.empty(1, in_channels))

        self.lin1 = Linear(in_channels + edge_dim, out_channels, False)
        self.lin2 = Linear(out_channels, out_channels, False)

        self.bias = Parameter(torch.empty(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.att_l)
        glorot(self.att_r)
        glorot(self.lin1.weight)
        glorot(self.lin2.weight)
        zeros(self.bias)

    def forward(self, x: Tensor, edge_index: Adj, edge_attr: Tensor) -> Tensor:
        alpha = self.edge_updater(edge_index, x=x, edge_attr=edge_attr)
        out = self.propagate(edge_index, x=x, alpha=alpha)
        out = out + self.bias
        return out

    def edge_update(self, x_j: Tensor, x_i: Tensor, edge_attr: Tensor,
                    index: Tensor, ptr: OptTensor,
                    size_i: Optional[int]) -> Tensor:
        x_j = F.leaky_relu_(self.lin1(torch.cat([x_j, edge_attr], dim=-1)))
        alpha_j = (x_j @ self.att_l.t()).squeeze(-1)
        alpha_i = (x_i @ self.att_r.t()).squeeze(-1)
        alpha = alpha_j + alpha_i
        alpha = F.leaky_relu_(alpha)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        return alpha

    def message(self, x_j: Tensor, alpha: Tensor) -> Tensor:
        return self.lin2(x_j) * alpha.unsqueeze(-1)


class ParaAttentiveFP(torch.nn.Module):
    r"""The Attentive FP model for molecular representation learning from the
    `"Pushing the Boundaries of Molecular Representation for Drug Discovery
    with the Graph Attention Mechanism"
    <https://pubs.acs.org/doi/10.1021/acs.jmedchem.9b00959>`_ paper, based on
    graph attention mechanisms.

    Args:
        in_channels (int): Size of each input sample.
        hidden_channels (int): Hidden node feature dimensionality.
        out_channels (int): Size of each output sample.
        edge_dim (int): Edge feature dimensionality.
        num_layers (int): Number of GNN layers.
        num_timesteps (int): Number of iterative refinement steps for global
            readout.
        dropout (float, optional): Dropout probability. (default: :obj:`0.0`)

    """
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        edge_dim: int,
        num_layers: int,
        num_timesteps: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        self.num_layers = num_layers
        self.num_timesteps = num_timesteps
        self.dropout = dropout

        self.lin1 = Linear(in_channels, hidden_channels)

        self.gate_conv = GATEConv(hidden_channels, hidden_channels, edge_dim,
                                  dropout)
        self.gru = GRUCell(hidden_channels, hidden_channels)

        self.atom_convs = torch.nn.ModuleList()
        self.atom_grus = torch.nn.ModuleList()
        for _ in range(num_layers - 1):
            conv = GATConv(hidden_channels, hidden_channels, dropout=dropout,
                           add_self_loops=False, negative_slope=0.01)
            self.atom_convs.append(conv)
            self.atom_grus.append(GRUCell(hidden_channels, hidden_channels))

        self.mol_conv1 = GATConv(hidden_channels, hidden_channels,
                                dropout=dropout, add_self_loops=False,
                                negative_slope=0.01)
        self.mol_conv1.explain = False  # Cannot explain global pooling.
        self.mol_gru1 = GRUCell(hidden_channels, hidden_channels)

        self.mol_conv2 = GATConv(hidden_channels, hidden_channels,
                                dropout=dropout, add_self_loops=False,
                                negative_slope=0.01)

        self.mol_conv2.explain = False  # Cannot explain global pooling.
        self.mol_gru2 = GRUCell(hidden_channels, hidden_channels)

        self.lin2 = Linear(hidden_channels, out_channels)

        self.reset_parameters()

    def reset_parameters(self):
        r"""Resets all learnable parameters of the module."""
        self.lin1.reset_parameters()
        self.gate_conv.reset_parameters()
        self.gru.reset_parameters()
        for conv, gru in zip(self.atom_convs, self.atom_grus):
            conv.reset_parameters()
            gru.reset_parameters()
        self.mol_conv1.reset_parameters()
        self.mol_gru1.reset_parameters()
        self.mol_conv2.reset_parameters()
        self.mol_gru2.reset_parameters()
        self.lin2.reset_parameters()

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor,
                batch: Tensor) -> Tensor:
        """"""  # noqa: D419
        # Atom Embedding:
        x = F.leaky_relu_(self.lin1(x))

        h = F.elu_(self.gate_conv(x, edge_index, edge_attr))
        h = F.dropout(h, p=self.dropout, training=self.training)
        x = self.gru(h, x).relu_()

        x_a = x.clone()
        x_m = x.clone()

        # Atom Embedding:
        for conv, gru in zip(self.atom_convs, self.atom_grus):
            h = conv(x_a, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            x_a = gru(h, x_a).relu()

        x_a_out = global_add_pool(x_a, batch).relu_()

        # Molecule Embedding:
        row = torch.arange(batch.size(0), device=batch.device)
        edge_index = torch.stack([row, batch], dim=0)


        out2 = x_a_out
        for t in range(self.num_timesteps):
            h2 = F.elu_(self.mol_conv1((x_a, out2), edge_index))
            h2 = F.dropout(h2, p=self.dropout, training=self.training)
            out2 = self.mol_gru1(h2, out2).relu_()

        return out2

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}('
                f'in_channels={self.in_channels}, '
                f'hidden_channels={self.hidden_channels}, '
                f'out_channels={self.out_channels}, '
                f'edge_dim={self.edge_dim}, '
                f'num_layers={self.num_layers}, '
                f'num_timesteps={self.num_timesteps}'
                f')')



class Solute_Solvent_Model(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim, num_layers, num_timesteps, dropout):
        super(Solute_Solvent_Model, self).__init__()
        # self.attentive_fp = ParaAttentiveFP(in_channels, hidden_channels, out_channels, edge_dim, num_layers, num_timesteps, dropout)
        self.attentive_fp1 = ParaAttentiveFP(in_channels, hidden_channels, out_channels, edge_dim, num_layers, num_timesteps, dropout)
        self.attentive_fp2 = ParaAttentiveFP(in_channels, hidden_channels, out_channels, edge_dim, num_layers, num_timesteps, dropout)
        self.projection_gnn = ProjectionMLP(hidden_channels, hidden_channels)
        self.projection_llm = ProjectionMLP(384, hidden_channels)
        self.CrossModalAttention = CrossModalAttention(hidden_channels, hidden_channels, hidden_channels)
        self.MergerGAT = GATConv(hidden_channels, hidden_channels)
        self.merger_gru = GRUCell(hidden_channels, hidden_channels)
        self.linear = torch.nn.Linear(hidden_channels, 1)


    def forward(self, combined_data):
        data_solute, data_solvent = combined_data
        x1, edge_index1, edge_attr1, batch1 = data_solute.x, data_solute.edge_index, data_solute.edge_attr, torch.zeros(data_solute.x.size(0), dtype=torch.long)
        x2, edge_index2, edge_attr2, batch2 = data_solvent.x, data_solvent.edge_index, data_solvent.edge_attr, torch.zeros(data_solvent.x.size(0), dtype=torch.long)

        x1 = self.attentive_fp1(x1, edge_index1, edge_attr1, batch1)
        x2 = self.attentive_fp2(x2, edge_index2, edge_attr2, batch2)

        # Combine x1 and x2 into a single graph
        x = torch.cat([x1, x2], dim=0)  # Two nodes in the graph

        # Create edge index to connect the two nodes
        edge_index = torch.tensor([[0, 1], [1, 0]], dtype=torch.long)

        # Add self-loops [0, 0] and [1, 1]
        self_loops = torch.tensor([[0, 0], [1, 1]], dtype=torch.long)

        # Concatenate the original edges and the self-loops
        edge_index = torch.cat([edge_index, self_loops], dim=1)

        # Pass the interaction graph to MergerGAT
        output = self.MergerGAT(x, edge_index)
        out = self.merger_gru(output, x).relu_()

        batch = torch.tensor([0, 0], dtype=torch.long)
        out = global_add_pool(out, batch)
        out = self.linear(out)
        return out

    def __repr__(self):
        return self.attentive_fp.__repr__()



#set up device and create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') #use CUDA if available
model = Solute_Solvent_Model(in_channels = 41, hidden_channels = 48, out_channels = 1, edge_dim = 10, num_layers = 3, num_timesteps = 3, dropout = 0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) #use Adam optimizer


mae_loss = torch.nn.L1Loss()


# # Training model
# model.train()  # Set model to training mode
# best_val_loss = float('inf')
# patience = 10  # Number of epochs to wait for improvement
# trigger_times = 0
# # torch.autograd.set_detect_anomaly(True)
# gamma = 100

# for epoch in range(50):  # Run for a defined number of epochs
#     sum_loss = 0  # Used to compute average loss in an epoch
#     num_correct = 0
#     # random.shuffle(train)  # Shuffle the training data each epoch

#     # Training loop
#     for d in tqdm(train_data):  # Go over each training point
#         data = d # Send data to device
#         optimizer.zero_grad()  # Zero gradients
#         out = model(data)  # Evaluate data point

#         loss = mae_loss(out.squeeze(), data[0].y.squeeze())
#         sum_loss += float(loss)  # Aggregate loss
#         loss.backward()  # Compute gradients
#         optimizer.step()  # Apply optimization

#     # Average loss and accuracy for the training set
#     avg_loss = sum_loss / len(train_data)
#     print('Epoch: {:03d}, Average MAE: {:.5f}'.format(epoch+1, avg_loss))

#     # Validation phase
#     model.eval()  # Set model to evaluation mode
#     val_loss = 0

#     with torch.no_grad():  # Disable gradient calculation
#         for d in val_data:
#             data = d  # Send data to device
#             out = model(data)  # Evaluate data point
#             loss = mae_loss(out.squeeze(), data[0].y.squeeze())
#             val_loss += float(loss)  # Aggregate validation loss


#     avg_val_loss = val_loss / len(val_data)
#     print('Validation MAE: {:.5f}'.format(avg_val_loss))

#     # Early stopping check
#     if avg_val_loss < best_val_loss:
#         best_val_loss = avg_val_loss
#         trigger_times = 0  # Reset trigger times
#         print('Model improved! Saving model...')
#         torch.save(model.state_dict(), 'gdrive/My Drive/Base_GNN_Solubility/New_Architecture_Exp_DB_w_o_CG.pth')  # Save the model
#     else:
#         trigger_times += 1
#         # print('No improvement.')

#     if trigger_times >= patience:
#         print('Early stopping triggered.')
#         break  # Stop training if no im provement

# Load the best model for testing
model.load_state_dict(torch.load('gdrive/My Drive/Base_GNN_Solubility/HASolGNN_Trained_Exp_DB.pth',  weights_only=True))
model.eval()

# Testing phase
test_loss = 0
test_correct = 0
all_actual = []
all_predicted = []

with torch.no_grad():
    for d in test_data:
        data = d
        out = model(data)  # Evaluate test point
        actual_values = data[0].y.squeeze().cpu().numpy()  # Get actual values
        predicted_values = out.squeeze().cpu().numpy()  # Get predicted values
        loss = mae_loss(out.squeeze(), data[0].y.squeeze())
        test_loss += float(loss)

        if actual_values.ndim == 0:
            all_actual.append(actual_values)
        else:
            all_actual.extend(actual_values)  # Add to the list of actual values

        if predicted_values.ndim == 0:
            all_predicted.append(predicted_values)
        else:
            all_predicted.extend(predicted_values)  # Add to the list of predicted values

avg_test_loss = test_loss / len(test_data)
print('Test MAE: {:.5f}'.format(avg_test_loss))
# Calculate R-squared
r2 = r2_score(all_actual, all_predicted)
print('R-squared: {:.5f}'.format(r2))

Test MAE: 0.69922
R-squared: 0.80438
